In [2]:
import pandas as pd
import numpy as np

Clientes se queda igual.
Coolers -> Cooler_unico
Sales_Churn_Test + Sales_Churn_Train -> Sales_historial_unico

In [3]:
# 1. Cargar el archivo original de coolers
df_coolers = pd.read_csv("coolers.csv")

# Asegurar los nombres de las columnas por posición (por si acaso)
# Índice 0: customer_id, Índice 2: num_coolers, Índice 3: num_doors
col_id = df_coolers.columns[0]
col_coolers = df_coolers.columns[2]
col_doors = df_coolers.columns[3]

print(f"Procesando {df_coolers.shape[0]:,} registros mensuales...")

# 2. Agrupar por cliente y calcular el promedio de las columnas numéricas
# Al no incluir 'calmonth' en el desglose, desaparece de la estructura final.
df_coolers_unico = (
    df_coolers.groupby(col_id)[[col_coolers, col_doors]]
    .mean()
    .reset_index()
)

# 3. Renombrar las columnas para que quede claro que ahora son promedios
df_coolers_unico = df_coolers_unico.rename(
    columns={
        col_coolers: "promedio_num_coolers",
        col_doors: "promedio_num_doors",
    }
)

# 4. Guardar el resultado en un nuevo archivo CSV
nombre_salida = "coolers_unico.csv"
df_coolers_unico.to_csv(nombre_salida, index=False)

print("\n=== PROCESO TERMINADO ===")
print(f"✨ Ahora tienes exactamente 1 fila por cliente.")
print(f"📊 Total de clientes únicos guardados: {df_coolers_unico.shape[0]:,}")
print(f"💾 Archivo guardado como: '{nombre_salida}'")

# Mostrar una pequeña muestra de cómo quedó
print("\n👇 Muestra del nuevo archivo:")
print(df_coolers_unico.head())

Procesando 4,636,676 registros mensuales...

=== PROCESO TERMINADO ===
✨ Ahora tienes exactamente 1 fila por cliente.
📊 Total de clientes únicos guardados: 208,024
💾 Archivo guardado como: 'coolers_unico.csv'

👇 Muestra del nuevo archivo:
                                         customer_id  promedio_num_coolers  \
0  00003bbeecfc76c50726de7508e265ab37ef665a882db0...              1.000000   
1  00008908998fbc054fa79e72be7b19357253fab910d570...              2.615385   
2  00008d60b3a3092f85e58a841c563a425c3ccc262c26a1...              1.000000   
3  000183038b1005f0a3c853b721d079c8bf8dc94d2d441d...              1.000000   
4  0001af045b2ccffd5b723a962c0dd5599083e6efb6d0a2...              2.000000   

   promedio_num_doors  
0            2.000000  
1            2.615385  
2            2.000000  
3            2.000000  
4            2.000000  


In [4]:
# 1. Configurar nombres de los archivos mensuales de entrada
archivo_train = "sales_churn_train.csv"
archivo_test = "sales_churn_test.csv"

print("📖 Leyendo archivos mensuales de Train y Test...")
df_train = pd.read_csv(archivo_train)
df_test = pd.read_csv(archivo_test)

# Asegurar las posiciones de las columnas (asumiendo que tienen la misma estructura)
col_id = df_train.columns[0]  # customer_id
col_trans = df_train.columns[2]  # num_transacciones
col_boxes = df_train.columns[3]  # uni_boxes_sold_m
col_target = (
    "Target" if "Target" in df_train.columns else df_train.columns[4]
)  # Target en Train

print(f"   -> Historial de Train original: {df_train.shape[0]:,} registros.")
print(f"   -> Historial de Test original: {df_test.shape[0]:,} registros.")

# ========================================================
# 🛠️ PASO PREVIO: LIMPIEZA DE NEGATIVOS EN TRAIN
# ========================================================
print("\n🧽 Aplicando reglas de limpieza sobre el archivo de Train...")

# Identificar registros con transacciones negativas O cajas negativas
condicion_negativos = (df_train[col_trans] < 0) | (df_train[col_boxes] < 0)

# Caso A: Negativos donde el Target es 0 (Se eliminan)
filas_a_eliminar = condicion_negativos & (df_train[col_target] == 0)
total_eliminadas = filas_a_eliminar.sum()

# Caso B: Negativos donde el Target es 1 (Se convierten a 0)
filas_a_modificar = condicion_negativos & (df_train[col_target] == 1)
total_modificadas = filas_a_modificar.sum()

# Ejecutar modificaciones en las columnas correspondientes
df_train.loc[filas_a_modificar & (df_train[col_trans] < 0), col_trans] = 0
df_train.loc[filas_a_modificar & (df_train[col_boxes] < 0), col_boxes] = 0

# Ejecutar eliminación quedándonos con lo que NO se debe borrar
df_train_limpio = df_train[~filas_a_eliminar].copy()

print(f"   🗑️  Filas eliminadas en Train (Negativos con Target 0): {total_eliminadas}")
print(
    f"   🛠️  Filas corregidas en Train (Negativos a 0 con Target 1): {total_modificadas}"
)
print(f"   📊 Train tras la limpieza: {df_train_limpio.shape[0]:,} registros.")

# ========================================================
# PASO COMPLEMENTARIO: CONCATENAR HISTORIALES
# ========================================================
print("\n🧲 Concatenando Train limpio y Test en una sola matriz...")
# Usamos el train limpio. ignore_index=True rehace el conteo de filas.
df_historial_completo = pd.concat([df_train_limpio, df_test], ignore_index=True)
print(
    f"   -> Total de registros combinados para promediar: {df_historial_completo.shape[0]:,} filas."
)

# ========================================================
# PASO 2: AGRUPAR POR CLIENTE Y PROMEDIAR
# ========================================================
print("\n🔄 Colapsando historial: calculando promedios por cliente...")
# Al agrupar, Pandas calculará la media real usando los ceros corregidos y sin las filas ruidosas
df_ventas_unico = (
    df_historial_completo.groupby(col_id)[[col_trans, col_boxes]]
    .mean()
    .reset_index()
)

# 3. Renombrar las columnas finales limpias
df_ventas_unico = df_ventas_unico.rename(
    columns={
        col_trans: "promedio_num_transacciones",
        col_boxes: "promedio_uni_boxes_sold_m",
    }
)

# 4. Guardar en un único archivo CSV consolidado
nombre_salida = "sales_historial_unico.csv"
df_ventas_unico.to_csv(nombre_salida, index=False)

print("\n=== ¡PROCESO CONSOLIDADO Y LIMPIO TERMINADO! ===")
print(f"✨ Estructura unificada a 1 fila por cliente.")
print(f"📊 Total de clientes únicos globales calculados: {df_ventas_unico.shape[0]:,}")
print(f"💾 Archivo guardado como: '{nombre_salida}'")

# Mostrar muestra de control
print("\n👇 Muestra del nuevo archivo unificado con promedios corregidos:")
print(df_ventas_unico.head())

📖 Leyendo archivos mensuales de Train y Test...
   -> Historial de Train original: 5,030,534 registros.
   -> Historial de Test original: 199,923 registros.

🧽 Aplicando reglas de limpieza sobre el archivo de Train...
   🗑️  Filas eliminadas en Train (Negativos con Target 0): 62
   🛠️  Filas corregidas en Train (Negativos a 0 con Target 1): 17
   📊 Train tras la limpieza: 5,030,472 registros.

🧲 Concatenando Train limpio y Test en una sola matriz...
   -> Total de registros combinados para promediar: 5,230,395 filas.

🔄 Colapsando historial: calculando promedios por cliente...

=== ¡PROCESO CONSOLIDADO Y LIMPIO TERMINADO! ===
✨ Estructura unificada a 1 fila por cliente.
📊 Total de clientes únicos globales calculados: 243,325
💾 Archivo guardado como: 'sales_historial_unico.csv'

👇 Muestra del nuevo archivo unificado con promedios corregidos:
                                         customer_id  \
0  0000443786ce90b386b37f64ae7ae011f9c2a71733855a...   
1  00008908998fbc054fa79e72be7b1935

Master maker:
Toma de entrada Preds_submission, Clientes, Coolers_unicos, Sales_historial_unico
Salida: Master prediction

In [ ]:
# 1. Definir los nombres de tus archivos CSV actualizados
archivo_prediction = "preds_submission.csv"  # El que manda (tiene target y customer_id)
archivo_clientes = "clientes.csv"
archivo_coolers = "coolers_unico.csv" 
archivo_ventas_historico = "sales_historial_unico.csv"  

try:
    print("📖 Cargando archivos...")
    # 2. Cargar el archivo base (el que manda)
    df_base = pd.read_csv(archivo_prediction)

    # Cargar los archivos complementarios consolidados
    df_clientes = pd.read_csv(archivo_clientes)
    df_coolers = pd.read_csv(archivo_coolers)
    df_ventas = pd.read_csv(archivo_ventas_historico)

    # ========================================================
    # 🛠️ CORRECCIÓN DE ESTRUCTURA EN EL ARCHIVO BASE
    # ========================================================
    # Detectamos los nombres reales por posición: índice 0 es Target, índice 1 es Customer ID
    col_target_original = df_base.columns[0]
    col_id = df_base.columns[1]  # Este es tu 'customer_id'

    print(f"🔄 Reestructurando '{archivo_prediction}'...")
    # Reordenamos las columnas para que el ID sea la primera y el target la segunda
    df_base = df_base[[col_id, col_target_original]]

    # Renombramos la columna target a 'prob_churn'
    df_base = df_base.rename(columns={col_target_original: "prob_churn"})
    # ========================================================

    # Renombrar preventivamente la primera columna de los otros archivos para asegurar el match
    df_clientes.rename(columns={df_clientes.columns[0]: col_id}, inplace=True)
    df_coolers.rename(columns={df_coolers.columns[0]: col_id}, inplace=True)
    df_ventas.rename(columns={df_ventas.columns[0]: col_id}, inplace=True)

    print("🔗 Uniendo los sets de datos (Manda el archivo de predicciones ordenado)...")

    # STEP 1: Unir con datos fijos de Clientes (Región, Canal, Tamaño)
    df_unificado = pd.merge(df_base, df_clientes, on=col_id, how="left")

    # STEP 2: Unir con la infraestructura promedio de Coolers (Coolers, Puertas)
    df_unificado = pd.merge(df_unificado, df_coolers, on=col_id, how="left")

    # STEP 3: Unir con el historial combinado de ventas (Transacciones y Cajas promedio)
    df_unificado = pd.merge(df_unificado, df_ventas, on=col_id, how="left")

    print("✨ Limpiando valores faltantes...")

    # STEP 4: Reemplazar cualquier celda vacía (NaN) por un 0 plano
    df_final = df_unificado.fillna(0)

    # 3. Guardar en el nuevo archivo consolidado maestro
    archivo_salida = "master_predictions_data.csv"
    df_final.to_csv(archivo_salida, index=False)

    print("\n=== PROCESO COMPLETADO EXCEPCIONALMENTE ===")
    print(f"📋 Filas originales en prediction: {df_base.shape[0]:,}")
    print(f"📊 Filas en el nuevo CSV maestro: {df_final.shape[0]:,}")
    print(f"💾 Archivo guardado con éxito como: '{archivo_salida}'")

    # Mostrar una pequeña muestra de cómo quedaron alineadas las columnas limpias
    print("\n👇 Muestra de las primeras filas consolidadas:")
    print(df_final.head())

except FileNotFoundError as e:
    print(
        f"\n❌ Error de archivo: Asegúrate de que todos los CSV estén en la misma carpeta. Detalle: {e}"
    )

📖 Cargando archivos...
🔄 Reestructurando 'preds_submission.csv'...
🔗 Uniendo los sets de datos (Manda el archivo de predicciones ordenado)...
✨ Limpiando valores faltantes...

=== PROCESO COMPLETADO EXCEPCIONALMENTE ===
📋 Filas originales en prediction: 199,923
📊 Filas en el nuevo CSV maestro: 199,923
💾 Archivo guardado con éxito como: 'master_predictions_data.csv'

👇 Muestra de las primeras filas consolidadas:
                                         customer_id  prob_churn  \
0  a75ba329134e79b76ea268a5613e9fac8f794523143d12...         0.0   
1  11d7eaae80505808b512ea0585a3ed142f57b6b8988c53...         0.0   
2  28edb943e8c4869dc04102667b5ede509db73aeb54e89d...         0.0   
3  d40f820f096646c130c91b3c9be257e4cb0bf75b88e82b...         0.0   
4  dad1eb4c0e38c3a6305ef4e2b328ab395ce033abf98a9c...         0.0   

       territory_d comercial_subchannel_d rtm_customer_size_d  \
0      Guadalajara               Farmacia                Mini   
1        Matamoros    Abarrotes y bodegas     

In [7]:
df_master = pd.read_csv("master_predictions_data.csv")
df_master.head(0)

,customer_id,prob_churn,territory_d,comercial_subchannel_d,rtm_customer_size_d,promedio_num_coolers,promedio_num_doors,promedio_num_transacciones,promedio_uni_boxes_sold_m
